# Fase 6 - Parte A: Revisión de anomalías en Imágenes

Este script se encarga de preparar la revisión de las anomalías de la fase 3 en Google Earth Pro. 
Genera dos archivos. 

1. Una planilla de Excel con los datos de cada anomalía y las verificaciones que hay que hacer al revisarla

2. Un archivo KML con las coronas, para ubicar cada anomalía en Google Earth Pro.


La revisión se hace de forma manual . La planilla calcula sola la lectura de cada anomalía a partir de las respuestas. El resultado de la planilla es usado por el script siguiente.


La revisión se hace mediante seis preguntas que se buscan responder observando el evento en GoogleEarth Pro. 

| Columna | Pregunta | Respuestas posibles |
|---|---|---|
| P1 | ¿Es un solo evento? | Sí, No, No se puede ver |
| P2 | ¿Incluye arranque y depósito? | Sí, No, No se puede ver |
| P3 | ¿Incluye algo ajeno (cauce, vía, otro depósito)? | Sí, No, No se puede ver |
| P4 | ¿La corona está en la parte alta de la cicatriz? | Sí, No, No se puede ver |
| P5 | Si hay un error, ¿empuja la métrica en el sentido de la anomalía? | Sí, No, No aplica (no hay error) |
| P6 | ¿Se ve en la imagen el rasgo que la hace anómala? | Sí, No, No se puede ver |


Se usa como guía de la revisión casos en los que cierto error de digitalización podrían producir el perfil de esa anomalía:

| Perfil | Error | 
|---|---|
| Más móvil (H/L menor) | El polígono incluye un cauce, una vía o el depósito de otro evento, que alargan L | 
| Área mayor | El polígono une varios eventos o una reactivación, o abarca más que la cicatriz. | 
| Menos alargado |El polígono está truncado (le falta parte del trayecto) o une eventos lado a lado. |


**La lectura automática**

La última columna tiene una fórmula que revisa las respuestas en este orden y se queda con la primera condición que se cumpla:

Sin respuestas: «Pendiente».
P5 = Sí: «Error de inventario».
El evento está bajo el umbral de completitud: «Posible sesgo de registro».
P6 = Sí: «Proceso real».
El polígono se digitalizó desde un punto de la fuente: «Posible sesgo de registro».
Ninguna de las anteriores: «Indeterminada».



# Librerías y configuración

### Qué hace

Carga las librerías y reúne las rutas y los textos que se escriben en la planilla.



In [ ]:
import os
from xml.sax.saxutils import escape

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

# -----------------------------------------------------------------------------
# CONFIGURACION 
# -----------------------------------------------------------------------------

# Carpetas de la Fase 3 y de la Fase 6.

CARPETA_F3 = r"D:\Users\tasal\Desktop\Tesis de grado\Fase 3"
CARPETA_F6 = r"D:\Users\tasal\Desktop\Tesis de grado\Fase 6"


# Salida final de la Fase 3 y tabla del inventario.

ARCH_INTEGRADO = os.path.join(CARPETA_F3, "F3_fase3_integrado.csv")
ARCH_INVENTARIO = os.path.join(CARPETA_F6, "tabla_inventario.xls")   

 
CAPA_POLIGONOS = None             # Capa de poligonos  para el KMZ (opcional)
EPSG_ORIGEN = 9377                # Sistema de coordenadas de la corona.

# Convierte el texto del perfil de la Fase 3 en una etiqueta corta.

PERFIL = {
    "H/L menor de lo esperado (mas movil)": "Más móvil (H/L menor)",
    "area mayor de lo esperado": "Área mayor",
    "menos alargado de lo esperado": "Menos alargado (elongación menor)",
}


# El error de digitalización que podría producir cada perfil.

ERROR_QUE_EXPLICA = {
    "Más móvil (H/L menor)": "El polígono incluye un cauce, una vía o el depósito de otro evento (alarga L); "
                             "o el pie quedó sobre un lleno",
    "Área mayor": "El polígono une varios eventos o una reactivación, o abarca más que la cicatriz",
    "Menos alargado (elongación menor)": "El polígono está truncado (falta parte del trayecto) o une eventos lado a lado",
}

# Funciones

1. `a_wgs84(x, y)` pasa las coordenadas de EPSG 9377 a longitud y latitud WGS84, que es lo que usa Google Earth.
2. `leer_inventario(ruta)`  lee la tabla del inventario en CSV o en Excel.


In [ ]:
def a_wgs84(x, y):
    """Convierte coordenadas de EPSG 9377 a longitud y latitud WGS84."""
    try:
        from pyproj import Transformer
        t = Transformer.from_crs(EPSG_ORIGEN, 4326, always_xy=True)
        return t.transform(x, y)
    except ImportError:
        import arcpy
        sr_o, sr_d = arcpy.SpatialReference(EPSG_ORIGEN), arcpy.SpatialReference(4326)
        lon, lat = [], []
        for xi, yi in zip(x, y):
            p = arcpy.PointGeometry(arcpy.Point(xi, yi), sr_o).projectAs(sr_d).firstPoint
            lon.append(p.X)
            lat.append(p.Y)
        return lon, lat


def leer_inventario(ruta):
    if ruta.lower().endswith(".csv"):
        return pd.read_csv(ruta, encoding="utf-8-sig")
    return pd.read_excel(ruta)

# Programa principal

### ¿Que hace?

Arma la planilla, el KML y, si se pidió, el KMZ.


**Como lo hace**

1. Lee la salida de la Fase 3 y el inventario, convierte la fecha de ocurrencia en fecha, extrae las anomalías y les une su fila del inventario por el identificador. Las ordena por distancia robusta, de mayor a menor, convierte las coordenadas de la corona y pone el perfil corto y el origen precargado.

2.  Escribe un encabezado de 25 columnas y una fila por anomalía con sus datos, más ocho columnas vacías para la revisión.

3.  Escribe un punto por corona, con el identificador como nombre y un resumen (nivel, perfil, fuente, tipo, año y área) como descripción. En KML los colores van en orden alfa, azul, verde, rojo.



In [ ]:
   


def main():
    d = pd.read_csv(ARCH_INTEGRADO, encoding="utf-8-sig")
    inv = leer_inventario(ARCH_INVENTARIO)
    inv["Fecha_ocurrencia"] = pd.to_datetime(inv["Fecha_ocurrencia"], errors="coerce")

    a = (d[d["ANOMALIA"] != "normal"]
         .merge(inv, left_on="ID", right_on="ID_evento", how="left")
         .sort_values("RD", ascending=False))
    a["lon"], a["lat"] = a_wgs84(a["X"].values, a["Y"].values)
    a["Perfil"] = a["PERFIL"].map(PERFIL)
    # Origen precargado; el autor lo corrigio en la planilla cuando la fuente entrego un poligono
    a["Origen"] = a["GRUPO"].map({"Institucional": "Punto de la fuente (digitalizado por mí)",
                                  "Mapeo": "Mapeo propio sobre imagen"})
    print(f"Anomalias: {len(a)}")

    # --- Planilla ---
    wb = Workbook()
    ws = wb.active
    ws.title = "Revision"
    hdr = ["ID", "Nivel", "Distancia robusta", "Perfil de la anomalía", "Error que la explicaría", "Grupo",
           "Fuente", "Tipo / subtipo", "Año", "Fecha de ocurrencia", "Área (m²)", "Bajo umbral",
           "Origen del polígono", "Observación del inventario", "Latitud corona", "Longitud corona",
           "Fecha de la imagen usada", "P1 ¿Un solo evento?", "P2 ¿Incluye arranque y depósito?",
           "P3 ¿Incluye algo ajeno (cauce, vía, otro depósito)?",
           "P4 ¿La corona está en la parte alta de la cicatriz?",
           "P5 Si hay error: ¿empuja la métrica en el sentido de la anomalía?",
           "P6 ¿Se ve en la imagen el rasgo que la hace anómala?",
           "Descripción breve (error o rasgo visto)", "Lectura (automática)"]
    ws.append(hdr)
    for i, (_, r) in enumerate(a.iterrows(), start=2):
        sub = r["Sub_tipo"] if isinstance(r["Sub_tipo"], str) else ""
        obs = r["Observaciones"] if isinstance(r["Observaciones"], str) else ""
        ws.append([int(r["ID"]), r["ANOMALIA"], round(float(r["RD"]), 2), r["Perfil"],
                   ERROR_QUE_EXPLICA[r["Perfil"]], r["GRUPO"], r["fuente"],
                   f"{r['TIPO']}{' / ' + sub if sub else ''}",
                   int(r["Año"]) if pd.notna(r["Año"]) else "",
                   r["Fecha_ocurrencia"].strftime("%Y-%m-%d") if pd.notna(r["Fecha_ocurrencia"]) else "",
                   round(float(r["Shape_Area"])), "Sí" if r["BAJO_UMBRAL"] == 1 else "No", r["Origen"],
                   obs.strip(), round(float(r["lat"]), 6), round(float(r["lon"]), 6)] + [""] * 8)
        ws.cell(row=i, column=25).value = (
            f'=IF(COUNTA(R{i}:W{i})=0,"Pendiente",IF(V{i}="Sí","Error de inventario",'
            f'IF(L{i}="Sí","Posible sesgo de registro",IF(W{i}="Sí","Proceso real",'
            f'IF(LEFT(M{i},5)="Punto","Posible sesgo de registro","Indeterminada")))))')
    n = len(a) + 1

    gris, amarillo, azul = (PatternFill("solid", fgColor=c) for c in ("EDEDED", "FFF2CC", "DDEBF7"))
    linea = Side(style="thin", color="BFBFBF")
    borde = Border(left=linea, right=linea, top=linea, bottom=linea)
    for c in range(1, len(hdr) + 1):
        h = ws.cell(row=1, column=c)
        h.font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
        h.fill = PatternFill("solid", fgColor="404040")
        h.alignment = Alignment(wrap_text=True, vertical="center", horizontal="center")
        h.border = borde
        for r in range(2, n + 1):
            cel = ws.cell(row=r, column=c)
            cel.font = Font(name="Arial", size=10)
            cel.border = borde
            cel.alignment = Alignment(wrap_text=True, vertical="top")
            cel.fill = amarillo if (c == 13 or 17 <= c <= 24) else (azul if c == 25 else gris)
    for i, w in enumerate([6, 9, 9, 16, 34, 12, 18, 22, 7, 12, 9, 8, 22, 30, 11, 11, 13, 11, 13, 15, 15, 17, 15, 34, 20], 1):
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.row_dimensions[1].height = 75
    ws.freeze_panes = "B2"
    v1 = DataValidation(type="list", formula1='"Sí,No,No se puede ver"', allow_blank=True)
    v2 = DataValidation(type="list", formula1='"Sí,No,No aplica (no hay error)"', allow_blank=True)
    v3 = DataValidation(type="list", formula1='"Punto de la fuente (digitalizado por mí),'
                                              'Polígono entregado por la fuente,Mapeo propio sobre imagen"')
    for v in (v1, v2, v3):
        ws.add_data_validation(v)
    for col in "RSTUW":
        v1.add(f"{col}2:{col}{n}")
    v2.add(f"V2:V{n}")
    v3.add(f"M2:M{n}")
    ruta_xlsx = os.path.join(CARPETA_F6, "F6_planilla_revision.xlsx")
    wb.save(ruta_xlsx)
    print("Planilla:", ruta_xlsx)

    # --- KML de coronas ---
    marcas = []
    for _, r in a.iterrows():
        desc = (f"Nivel: {r['ANOMALIA']} | Perfil: {r['Perfil']} | {r['GRUPO']} ({r['fuente']}) | {r['TIPO']} | "
                f"Año: {int(r['Año']) if pd.notna(r['Año']) else 's.d.'} | Área: {r['Shape_Area']:,.0f} m²")
        estilo = "f" if r["ANOMALIA"] == "fuerte" else "m"
        marcas.append(f"<Placemark><name>{int(r['ID'])}</name><description>{escape(desc)}</description>"
                      f"<styleUrl>#{estilo}</styleUrl><Point><coordinates>{r['lon']:.7f},{r['lat']:.7f},0"
                      f"</coordinates></Point></Placemark>")
    icono = "<Icon><href>http://maps.google.com/mapfiles/kml/shapes/placemark_circle.png</href></Icon>"
    kml = ('<?xml version="1.0" encoding="UTF-8"?><kml xmlns="http://www.opengis.net/kml/2.2"><Document>'
           "<name>Coronas de las 19 anomalías (Fase 6)</name>"
           f'<Style id="f"><IconStyle><color>ff0000ff</color>{icono}</IconStyle></Style>'
           f'<Style id="m"><IconStyle><color>ff00a5ff</color>{icono}</IconStyle></Style>'
           + "".join(marcas) + "</Document></kml>")
    ruta_kml = os.path.join(CARPETA_F6, "coronas_19.kml")
    with open(ruta_kml, "w", encoding="utf-8") as f:
        f.write(kml)
    print("KML:", ruta_kml)

    # --- KMZ de poligonos (opcional, requiere arcpy y la capa de poligonos) ---
    if CAPA_POLIGONOS:
        import arcpy
        ids = ", ".join(str(int(i)) for i in a["ID"])
        arcpy.management.MakeFeatureLayer(CAPA_POLIGONOS, "anomalias_19", f"ID_evento IN ({ids})")
        ruta_kmz = os.path.join(CARPETA_F6, "anomalias_19.kmz")
        arcpy.conversion.LayerToKML("anomalias_19", ruta_kmz)
        print("KMZ:", ruta_kmz)


if __name__ == "__main__":
    main()